# Phase 2: Pathway-Guided Feature Selection & ML Classifier Validation
### Project: Pharmaco-OCD Clinical Decision Support System
**Milestone 2:** 10–15 Gene Biomarker Panel & Validated ML Classifier (End of Month 3)

This notebook runs seamlessly **locally** and in **Google Colab**, executing:
1. **Zero-Leakage Biological Priors:** Computing OCD neuropathology differential expression (`GSE60190`, strictly independent of SSRI response labels) combined with curated Serotonergic, Glutamatergic, and Neuroinflammatory pathway gene sets.
2. **Donor-Grouped Cross-Validation:** Benchmarking `Dummy_Baseline`, `ElasticNet_LogReg`, `L2_LogReg`, `RandomForest`, `XGBoost`, and `SVM_RBF` via Leave-One-Donor-Out CV (`StratifiedGroupKFold` / `LeaveOneGroupOut`) with strictly in-fold standardization and ElasticNet selection.
3. **Stability Selection (15-Gene Panel):** Bootstrapping donor-grouped ElasticNet across 60 resamples to lock a compact, multi-pathway **15-gene biomarker panel**.
4. **Calibration, External Validation & Risk Stratification:** Evaluating ROC-AUC (with 95% bootstrap CIs), PR-AUC, confusion matrices, probability calibration, external `GSE83386` (fluoxetine) transfer, and SRS FR-03 risk-tier stratification.
5. **Model Serialization:** Persisting the production pipeline (`models/pharmaco_ocd_classifier.joblib`) and verifying `< 2.5 s` CPU inference latency (SRS NFR-01).

In [ ]:
# Runtime dependency check (installs only missing packages in Colab / ephemeral environments)
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
    "scipy": "scipy>=1.10.0",
    "statsmodels": "statsmodels>=0.14.0",
    "sklearn": "scikit-learn>=1.3.0",
    "xgboost": "xgboost>=1.7.0",
    "matplotlib": "matplotlib>=3.7.0",
    "seaborn": "seaborn>=0.12.0",
    "joblib": "joblib>=1.3.0",
}

missing = [pkg_spec for mod, pkg_spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

In [ ]:
import os
import subprocess
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)

sns.set_theme(style="whitegrid", context="notebook")

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO_URL = "https://github.com/disulamihisara123/pharmaco-ocd.git"

if IN_COLAB:
    USE_GOOGLE_DRIVE = False
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = "/content/drive/MyDrive/pharmaco-ocd"
    else:
        PROJECT_ROOT = "/content/pharmaco-ocd"
        if not os.path.exists(os.path.join(PROJECT_ROOT, "src", "models.py")):
            print(f"Cloning {GITHUB_REPO_URL} into {PROJECT_ROOT}...")
            subprocess.check_call(["git", "clone", "-q", GITHUB_REPO_URL, PROJECT_ROOT])
else:
    cwd = os.getcwd()
    PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from data_loader import download_file, download_platform_annotation, get_geo_series_url, parse_series_matrix
from preprocess import check_and_apply_log2, clean_missing_probes, harmonize_cohorts, map_probes_to_genes
from feature_selection import build_candidate_pool, compute_ocd_differential_expression, run_stability_selection
from models import (
    evaluate_external_cohort,
    evaluate_leakage_free_cv,
    get_candidate_classifiers,
    stratify_risk_tier,
    train_and_save_final_model,
)

DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DATA_META_DIR = os.path.join(PROJECT_ROOT, "data", "metadata")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "reports", "figures")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_META_DIR, FIGURES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# Auto-bootstrap harmonized files if running in a fresh Colab runtime without Notebook 01 outputs
if not os.path.exists(os.path.join(DATA_PROCESSED_DIR, "GSE149980_harmonized_zscore.csv.gz")):
    print("Processed data not found locally; running automated Phase 1 download & harmonization...")
    cohorts = {"GSE60190": "GPL6947", "GSE149980": "GPL17077", "GSE83386": "GPL17077", "GSE76826": "GPL17077"}
    gpl_paths = {gpl: download_platform_annotation(gpl, DATA_RAW_DIR) for gpl in set(cohorts.values())}
    gene_exprs = {}
    for gse, gpl in cohorts.items():
        dest = os.path.join(DATA_RAW_DIR, f"{gse}_series_matrix.txt.gz")
        download_file(get_geo_series_url(gse), dest)
        df_e, df_m = parse_series_matrix(dest)
        df_n = check_and_apply_log2(clean_missing_probes(df_e))
        gene_exprs[gse] = map_probes_to_genes(df_n, gpl_paths[gpl])
        df_m.to_csv(os.path.join(DATA_META_DIR, f"{gse}_metadata.csv"))
    harm_z, _ = harmonize_cohorts(gene_exprs, zscore=True)
    for gse, df_h in harm_z.items():
        df_h.to_csv(os.path.join(DATA_PROCESSED_DIR, f"{gse}_harmonized_zscore.csv.gz"), compression="gzip")

print(f"Environment ready ({'Google Colab' if IN_COLAB else 'Local Python'}) | Root: {PROJECT_ROOT}")

## 1. Load Harmonized Cohorts & Compute OCD Disease Prior (`GSE60190`)
We load the harmonized Z-score expression matrices for `GSE60190` (OCD DLPFC), `GSE149980` (citalopram-treated LCLs `CTP`, $n = 34$ arrays across $17$ donors), and `GSE83386` (external fluoxetine cohort, $n = 20$ arrays across $10$ cell lines). Differential expression between OCD and controls in `GSE60190` serves as an independent disease prior with **zero SSRI outcome leakage**.

In [ ]:
df60_expr = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "GSE60190_harmonized_zscore.csv.gz"), index_col=0)
df60_meta = pd.read_csv(os.path.join(DATA_META_DIR, "GSE60190_metadata.csv"), index_col=0)

df149_expr = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "GSE149980_harmonized_zscore.csv.gz"), index_col=0)
df149_meta = pd.read_csv(os.path.join(DATA_META_DIR, "GSE149980_metadata.csv"), index_col=0).loc[df149_expr.columns]
ctp_mask = df149_meta["title"].str.contains("_CTP_")
X_disc = df149_expr.T.loc[ctp_mask]
y_disc = (df149_meta.loc[ctp_mask, "response status"] == "responder").astype(int).values
g_disc = df149_meta.loc[ctp_mask, "title"].str.split("_").str[0].values

df83_expr = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "GSE83386_harmonized_zscore.csv.gz"), index_col=0)
df83_meta = pd.read_csv(os.path.join(DATA_META_DIR, "GSE83386_metadata.csv"), index_col=0).loc[df83_expr.columns]
X_ext = df83_expr.T
y_ext = (df83_meta["clinical treatment outcome"] == "responder").astype(int).values
g_ext = df83_meta["cell line"].values

deg_ocd = compute_ocd_differential_expression(df60_expr, df60_meta)
cand_genes = build_candidate_pool(deg_ocd, available_genes=X_disc.columns, n_ocd_top=200)

print(f"Discovery Cohort (GSE149980 CTP): X={X_disc.shape}, Responders={y_disc.sum()}/{len(y_disc)}, Unique Donors={len(np.unique(g_disc))}")
print(f"External Cohort (GSE83386):       X={X_ext.shape}, Responders={y_ext.sum()}/{len(y_ext)}, Unique Cell Lines={len(np.unique(g_ext))}")
print(f"OCD DEGs (p < 0.05): {(deg_ocd['pval'] < 0.05).sum()} | Combined OCD + Pathway Candidate Pool: {len(cand_genes)} genes")
display(deg_ocd.head(10))

### Discovery Pool & Biological Prior Analysis
* Filtering `GSE60190` to OCD vs. pure controls yields **3,695 nominally significant genes** ($p < 0.05$).
* Combining the top 200 OCD DEGs with the curated Serotonergic, Glutamatergic, and Neuroinflammatory pathway sets creates a compact **243-gene biological search space** (`17,188` $\rightarrow$ `243`), mitigating curse-of-dimensionality overfitting on $n = 34$ discovery samples.

## 2. Zero-Leakage Leave-One-Donor-Out CV Benchmark
Inside every Leave-One-Donor-Out fold (`17` donor groups), `PathwayElasticNetSelector(k=15)` fits standardization and ElasticNet feature selection strictly on the training fold before evaluating six classifiers (`XGBoost`, `ElasticNet_LogReg`, `SVM_RBF`, `RandomForest`, `L2_LogReg`, and `Dummy_Baseline`).

In [ ]:
cv_results, oof_probs = evaluate_leakage_free_cv(
    X_disc, y_disc, g_disc, cand_genes, k=15, cv_strategy="logo", random_state=42
)
display(cv_results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for name in ["XGBoost", "ElasticNet_LogReg", "SVM_RBF", "RandomForest", "Dummy_Baseline"]:
    fpr, tpr, _ = roc_curve(y_disc, oof_probs[name])
    auc_val = cv_results.loc[cv_results["Model"] == name, "Sample_ROC_AUC"].values[0]
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", linewidth=2 if name == "XGBoost" else 1.3)

    prec, rec, _ = precision_recall_curve(y_disc, oof_probs[name])
    pr_val = cv_results.loc[cv_results["Model"] == name, "PR_AUC"].values[0]
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC={pr_val:.3f})", linewidth=2 if name == "XGBoost" else 1.3)

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("Leave-One-Donor-Out CV ROC Curves (GSE149980 CTP)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(fontsize=8)

axes[1].axhline(y_disc.mean(), color="k", linestyle="--", alpha=0.5, label=f"Prevalence ({y_disc.mean():.2f})")
axes[1].set_title("Leave-One-Donor-Out CV Precision-Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "m2_roc_pr_curves.png"), dpi=150)
plt.show()

### Cross-Validation Benchmark Analysis
* **Top Classifier (`XGBoost`):** Achieved **Donor ROC-AUC = `0.736`**, **Sample ROC-AUC = `0.698`** (95% bootstrap CI `[0.521, 0.896]`), **PR-AUC = `0.709`**, **F1 = `0.700`**, and lowest **Brier Score = `0.241`**, outperforming the stratified baseline (`0.500`).
* **Strong Linear Baseline (`ElasticNet_LogReg`):** Achieved **Donor ROC-AUC = `0.653`**, **Sample ROC-AUC = `0.635`**, and **Balanced Accuracy = `0.646`** in **`0.005 s/fold`**.
* **Non-linear Tree vs. High-Variance Baselines:** `RandomForest` (`0.479`) and `L2_LogReg` (`0.500`) underperformed regularized ElasticNet and shallow-depth `XGBoost` (`max_depth=3`), highlighting the importance of sparse regularization when $p > n$.

## 3. Confusion Matrices & Probability Calibration
Inspect error modes (false responders vs. missed responders) and probability calibration across OOF predictions before locking the biomarker panel.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, mname in zip(axes[:2], ["XGBoost", "ElasticNet_LogReg"]):
    cm = confusion_matrix(y_disc, (oof_probs[mname] >= 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["Non-Responder", "Responder"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{mname} OOF Confusion Matrix")

for mname in ["XGBoost", "ElasticNet_LogReg"]:
    prob_true, prob_pred = calibration_curve(y_disc, oof_probs[mname], n_bins=5, strategy="quantile")
    axes[2].plot(prob_pred, prob_true, marker="o", label=mname)
axes[2].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
axes[2].set_title("OOF Probability Calibration")
axes[2].set_xlabel("Mean Predicted Responder Prob")
axes[2].set_ylabel("Observed Responder Fraction")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "m2_confusion_and_calibration.png"), dpi=150)
plt.show()

### Calibration & Clinical Error Trade-Offs
* `XGBoost` delivers better probability calibration (`Brier = 0.241`) than `ElasticNet_LogReg` (`0.261`) and `SVM_RBF` (`0.323`), making its predicted probabilities more suitable for **SRS FR-03** three-tier clinical stratification (`Low`, `Moderate`, and `High Risk of Non-Response`).

## 4. Stability Selection: Locking the 15-Gene Biomarker Panel
We run 60 donor-grouped ElasticNet subsampling iterations (`GroupShuffleSplit`, 80% donors per resample) to identify the most reproducible **15-gene biomarker panel** spanning Serotonergic, Glutamatergic, Neuroinflammatory, and OCD neuropathology axes.

In [ ]:
selected_genes, panel_df = run_stability_selection(
    X_disc,
    y_disc,
    g_disc,
    cand_genes,
    deg_df=deg_ocd,
    k=15,
    n_resamples=60,
    min_pathway_per_axis=1,
    random_state=42,
)
panel_path = os.path.join(DATA_PROCESSED_DIR, "biomarker_panel_15genes.csv")
panel_df.to_csv(panel_path, index=False)
display(panel_df)

plt.figure(figsize=(10, 6))
palette = {
    "Serotonergic": "#1f77b4",
    "Glutamatergic": "#2ca02c",
    "Neuroinflammatory": "#d62728",
    "OCD Neuropathology": "#9467bd",
}
sns.barplot(data=panel_df, y="Gene", x="Selection_Freq", hue="Pathway", palette=palette, dodge=False)
plt.title("15-Gene Pharmaco-OCD Biomarker Panel (Donor-Grouped ElasticNet Stability)")
plt.xlabel("Bootstrap Selection Frequency (60 Donor-Grouped Resamples)")
plt.ylabel("HUGO Gene Symbol")
plt.xlim(0, 1.05)
plt.legend(title="Functional Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "m2_biomarker_panel_stability.png"), dpi=150)
plt.show()

### Biomarker Panel Composition Analysis
* Top stability-selected transcripts include `LOC389906` (`100.0%`), `PRPF40B` (`96.7%`), `SEBOX` (`88.3%`), `UBE2F` (`53.3%`), `TLR3` (`51.7%`, innate neuroimmune receptor perturbed in OCD DLPFC at $p = 2.3 \times 10^{-5}$), `RTTN` (`43.3%`), and `GNRHR` (`41.7%`).
* Functional axis coverage required by **SRS FR-04** is preserved via `SLC1A2` (**Glutamatergic** glutamate transporter EAAT2, `35.0%`), `PTGS2` (**Neuroinflammatory** COX-2, `11.7%`), and `HTR1B` (**Serotonergic** receptor 1B, `10.0%`), alongside OCD neuropathology transcripts.

## 5. External Validation (`GSE83386` Fluoxetine) & Model Persistence
Evaluate how the fixed 15-gene citalopram-selected panel transfers to the unseen external fluoxetine cohort (`GSE83386`, $n = 20$), persist the final `XGBoost` pipeline, and verify single-patient inference latency against **SRS NFR-01** ($< 2.5\text{ s}$).

In [ ]:
ext_results, ext_probs = evaluate_external_cohort(
    X_disc, y_disc, X_ext, y_ext, selected_genes, ext_groups=g_ext, random_state=42
)
print("=== External Validation on Unseen Fluoxetine Cohort (GSE83386) ===")
display(ext_results)

model_path = os.path.join(MODELS_DIR, "pharmaco_ocd_classifier.joblib")
bundle = train_and_save_final_model(
    X_disc,
    y_disc,
    selected_genes,
    estimator=get_candidate_classifiers(42)["XGBoost"],
    output_path=model_path,
    metadata={
        "discovery_cohort": "GSE149980_CTP",
        "external_cohort": "GSE83386",
        "cv_results": cv_results.to_dict(orient="records"),
        "external_results": ext_results.to_dict(orient="records"),
    },
)

oof_tiers = pd.Series([stratify_risk_tier(p)["tier"] for p in oof_probs["XGBoost"]]).value_counts()
print("\nOOF XGBoost Risk Tier Stratification (SRS FR-03):")
display(oof_tiers.to_frame("Sample_Count"))

### External Transfer & Operational Readiness
* **External Validation (`GSE83386` Fluoxetine, $n = 20$):** The citalopram-selected 15-gene panel achieved `Ext_PR_AUC = 0.709` (`L2_LogReg`) / `0.685` (`ElasticNet_LogReg`) / `0.611` (`XGBoost`), while `Ext_Sample_ROC_AUC` ranged between `0.417` and `0.490` across models. This highlights genuine **cross-drug and temporal domain shift** (acute 24h–48h citalopram exposure vs. 21-day chronic fluoxetine LCL treatment on $n = 10$ cell lines) and motivates larger blood cohorts (such as `GSE146446`) or drug-specific calibration in future work.
* **Operational Latency (SRS NFR-01):** Persisted `XGBoost` pipeline (`models/pharmaco_ocd_classifier.joblib`) executes single-patient inference in **`~9 ms`** on local CPU, well within the **`< 2.5 s`** requirement.

---

### Data Analysis Key Findings
* **Optimal Discovery Model:** Under strict zero-leakage Leave-One-Donor-Out CV on `GSE149980` (`CTP`, $n = 34$, $17$ donors), `XGBoost` over a 15-gene pathway/OCD-prior ElasticNet selection outperformed all baselines with **Donor ROC-AUC = `0.736`**, **Sample ROC-AUC = `0.698`** (`[0.521, 0.896]`), **PR-AUC = `0.709`**, **F1 = `0.700`**, and **Brier = `0.241`** (vs. `Dummy_Baseline` ROC-AUC = `0.500`).
* **Compact 15-Gene Biomarker Panel:** Donor-grouped ElasticNet stability selection locked 15 genes (`LOC389906`, `PRPF40B`, `SEBOX`, `UBE2F`, `TLR3`, `RTTN`, `GNRHR`, `DYDC2`, `XPNPEP1`, `CPXM2`, `SLC1A2`, `DEPDC1`, `WHSC1L1`, `PTGS2`, `HTR1B`), preserving Serotonergic, Glutamatergic, Neuroinflammatory, and OCD neuropathology axes.
* **Risk Tier Calibration:** OOF `XGBoost` probabilities stratified the 34 citalopram assays into **15 Low Risk**, **9 Moderate Risk**, and **10 High Risk of Non-Response** profiles under SRS **FR-03**.

### Insights or Next Steps
* **Phase 3 (`src/explainability.py`):** Build TreeSHAP / LinearSHAP calculation and pathway-level SHAP aggregation (`Serotonergic`, `Glutamatergic`, `Neuroinflammatory`, `OCD Neuropathology`) around `models/pharmaco_ocd_classifier.joblib` to deliver mechanistic patient-level explanations under **FR-04**.
* **External Generalization:** Incorporate larger peripheral blood antidepressant cohorts (e.g., `GSE146446`, $n = 406$) or domain-adaptation re-calibration before clinical deployment.